# 04 · Agent Observability with Langfuse

Unlike traditional deterministic software, agentic AI systems produce non-deterministic, multi-step, and multi-agent behaviors that shift the operational question from "is it up?" to "is it right?".
[(IBM, Agent Observability and Operations)](https://www.ibm.com/downloads/documents/us-en/1443d5dd174f42e6)

Observability is what lets you answer that question: capturing rich telemetry, ensuring traceability, and managing correctness, safety, and resilience in production.

This notebook demonstrates how to capture traces of AI agents using **Langfuse** for observability and tracing, and how to read a trace. It reuses the function-calling agent built in [`01_function_calling.ipynb`](01_function_calling.ipynb) rather than redefining it, and ships a pre-captured trace ([`trace.json`](trace.json)) so Step 4 (reading a trace as JSON) works even without Langfuse or network access set up yet.

1. **Langfuse Setup** - Installing and configuring Langfuse for tracing
2. **Agent Implementation** - Reusing the function-calling agent from 01
3. **Instrumenting your Agent with Langfuse Tracing** - Capture a trace with callback handler.
4. **Collect and read a trace** - Collect and read a trace.
5. **Conclusion**
6. **Advanced topics (⭐ stretch)** - Create a langfuse experiment - Capture a trace manually from langgraph events


# Steps


You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md).

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langfuse \
    langchain \
    langgraph \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    pandas \
    matplotlib \
    rich \
    rouge-score
! echo "::endgroup::"

### Import Dependencies

In [ ]:
import sys
from pathlib import Path

# Make the reusable `granite_agent` package (built in 01_function_calling) importable.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "granite_agent").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import json
import os
import tempfile
import time
import uuid
from typing import Any, Optional

from ibm_granite_community.notebook_utils import get_env_var
from langchain_core.messages import HumanMessage
from langfuse import Evaluation, Langfuse, get_client
from langfuse.langchain import CallbackHandler
from langgraph.graph.state import CompiledStateGraph
from rich import print
from rouge_score import rouge_scorer

from granite_agent.agent import State, build_agent, run_agent
from granite_agent.model import get_llm
from granite_agent.tools import get_current_weather, get_stock_price

### Setting up Langfuse

Langfuse gives you full visibility into your agent's behavior.

This lab uses a **self-hosted Langfuse at `http://localhost:3000`** (see the lab guide's familiarization walkthrough, step 5). On your own machine, or in Colab where nothing is running on `localhost`, use [Langfuse Cloud](https://us.cloud.langfuse.com/) instead -- everything below works the same either way, only `LANGFUSE_HOST` changes.

#### Connecting to Langfuse

Once you have access, grab your API credentials:

1. Go to `http://localhost:3000` (or your Langfuse Cloud project), sign up / sign in, and create a project.
2. Go to **Settings → API Keys** and generate a new key pair.
3. Copy your **Public Key**, **Secret Key**, and **Host URL** into your `.env` file:

```dotenv
LANGFUSE_SECRET_KEY=sk-lf-xxxxx
LANGFUSE_PUBLIC_KEY=pk-lf-xxxxx
LANGFUSE_HOST=http://localhost:3000
```

You are now ready to initialize the Langfuse client and start capturing agent traces.

## 2. Agent Implementation

### Set up a Granite AI model instance and reuse the agent from 01

This reuses the exact `get_llm()` / `build_agent()` functions built in [`01_function_calling.ipynb`](01_function_calling.ipynb) -- Ollama first, Replicate as the hosted fallback -- instead of hardcoding a single backend.

In [ ]:
llm = get_llm()
model_path = getattr(llm, "model", get_env_var("GRANITE_MODEL", "granite4.2:3b"))
print(f"Connected via {type(llm).__name__}, model={model_path!r}")

### The tools

`get_stock_price` and `get_current_weather` are the exact same tools from `01_function_calling.ipynb`, imported from `granite_agent.tools` rather than redefined here. These functions can use real web APIs if you obtain the necessary API keys (`AV_STOCK_API_KEY`, `WEATHER_API_KEY`); without them, they respond with a fixed, predetermined value for demonstration purposes.

### Create the agent

LangChain's `create_agent` builds a function-calling agent from a model and a list of tools. `granite_agent.agent.build_agent()` (from 01) wraps that exact call, so we reuse it instead of calling `create_agent` again here. For a detailed walkthrough of building this agent from scratch with LangGraph, see [`01_function_calling.ipynb`](01_function_calling.ipynb).

In [ ]:
tools = [get_stock_price, get_current_weather]

agent: CompiledStateGraph = build_agent(llm, tools)

Let's verify the agent works with a simple query, using the `run_agent` helper also built in 01:

In [ ]:
run_agent(agent, "What is the weather in Miami?")

## 3. Instrumenting your Agent with Langfuse

#### Initialize Langfuse Client

In [ ]:
langfuse_client = Langfuse(
    public_key=get_env_var("LANGFUSE_PUBLIC_KEY", "unset"),
    secret_key=get_env_var("LANGFUSE_SECRET_KEY", "unset"),
    host=get_env_var("LANGFUSE_HOST", "unset")
)

### What is a trace ?

A Langfuse trace represents a single request or operation in your AI application.

It captures the entire lifecycle of an execution, from the initial input to the final output, along with all intermediate steps and metadata.

### Observations

Each trace contains multiple observations that log individual steps of execution.
Observations provide granular visibility into what happens during a request, enabling detailed debugging and performance optimization.

They automatically nest through OpenTelemetry context propagation - each new observation becomes a child of the currently active one.

### Observation Types

| Type | Purpose |
|------|---------|
| **event** | Discrete point-in-time occurrences |
| **span** | Operations with duration |
| **generation** | AI model calls (prompts, tokens, costs) |
| **agent** | LLM-guided application flow decisions |
| **tool** | External API/service calls |
| **chain** | Links between application steps |
| **retriever** | Data retrieval (vector stores, databases) |
| **evaluator** | Assessment of LLM output quality |
| **embedding** | Embedding generation with metrics |
| **guardrail** | Protection against malicious content |


### Tracing with Langfuse CallbackHandler

1. We initialize a trace ID upfront, so we can link back to this specific trace later on.

2. We initialize Langfuse CallbackHandler with the trace ID

3. Then we Pass the Langfuse CallbackHandler via the config parameter to automatically capture the full execution trace of the function calling agent LangGraph agent created earlier.

In [ ]:
trace_id = langfuse_client.create_trace_id(seed=str(uuid.uuid4()))

langfuse_handler = CallbackHandler(trace_context={"trace_id": trace_id})

user_input = "What is the weather in Miami?"

config = {"callbacks": [langfuse_handler]}
input_state = State(messages=[user_input])
result = agent.invoke(input_state, config=config)

print(result.get('messages')[-1].content)

In [ ]:
trace_id

### Review Langfuse trace in Langfuse UI

The trace view shows the full execution tree of your LangGraph agent.

Key things to note:

- granite LLM calls are represented by generation nodes. Click into one to see the system prompt, user message, tools and response in the right-hand panel. The panel header shows the model name, token counts, and latency.
- When the model invokes a tool, it appears as a separate tools node in the trace.

**Note:** Tools are listed in Langfuse under role: `tool`
 but this is a UI representation only — they are not sent to the model in this format. The tools are sent to the model as a list of dictionaries, which is the expected format.

*(Open `http://localhost:3000` -- or the Langfuse Cloud URL printed by the cell above -- to see this trace for yourself; a screenshot isn't reproduced here since the tree looks slightly different for every model/tool combination.)*

## 4. Collect and Read a Langfuse Trace

In [ ]:
SAMPLE_TRACE_PATH = Path("trace.json")  # shipped with this notebook, no network needed
LIVE_TRACE_PATH = Path(tempfile.gettempdir()) / "granite_agent_trace.json"  # this session's own trace, if captured


def save_as_json(data: dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=4, default=str), encoding="utf-8")


def _parse_io(value):
    # The v2 observations API always returns input/output/metadata as raw JSON strings.
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def extract(trace_id: str, path: Path, *, timeout_s: float = 40.0, poll_interval_s: float = 3.0) -> None:
    # Self-hosted Langfuse v4 defaults to "events_only" write mode, which has no trace-level
    # read model: `langfuse.api.trace.get()` (the v3 API) 404s with "This endpoint is not
    # available ... events_only mode". The v4-native replacement is the v2 observations API,
    # which returns the same data as a flat list of observations instead of a Trace object.
    # See: https://langfuse.com/docs/api-and-data-platform/features/observations-api
    langfuse = get_client()

    # flush() only guarantees the spans were *delivered*, not that they're *queryable* yet --
    # server-side ingestion is asynchronous and can lag 15-30s (longer under load), during
    # which get_many() legitimately returns an empty list rather than raising. Poll for it.
    langfuse.flush()

    deadline = time.monotonic() + timeout_s
    observations = []
    while True:
        observations = langfuse.api.observations.get_many(
            trace_id=trace_id,
            fields="core,basic,io,metadata,model,usage,metrics,trace_context",
            limit=1000,
        ).data
        if observations or time.monotonic() >= deadline:
            break
        time.sleep(poll_interval_s)

    if not observations:
        raise ValueError(f"No observations found for trace {trace_id} after waiting {timeout_s:.0f}s for ingestion")

    # Exclude ChannelWrite observations.
    # If ChannelWrite are desired, simply set "exclude_channel_write = False" bellow

    # Information : ChannelWrite nodes are used internally by LangGraph to
    #               manage the flow of information and updates between different
    #               nodes or agents within a graph

    exclude_channel_write = True

    if exclude_channel_write:
        filtered = [obs for obs in observations if "ChannelWrite" not in (obs.name or "")]
        observations = filtered or observations

    # There is no more standalone "trace" object to read `input`/`output` from: the root
    # observation (the whole LangGraph run) now carries what used to be the trace-level I/O.
    root = next((obs for obs in observations if obs.is_root_observation), observations[0])

    observation_dicts = []
    for obs in observations:
        obs_dict = json.loads(obs.json())
        for key in ("input", "output", "metadata"):
            obs_dict[key] = _parse_io(obs_dict.get(key))
        observation_dicts.append(obs_dict)

    trace_data = {
        "id": trace_id,
        "name": root.trace_name or root.name,
        "timestamp": root.start_time.isoformat() if root.start_time else None,
        "latency": root.latency,
        "input": _parse_io(root.input),
        "output": _parse_io(root.output),
        "observations": observation_dicts,
    }

    save_as_json(trace_data, path)


try:
    print(f"Extracting for trace {trace_id}")
    extract(trace_id, LIVE_TRACE_PATH)
    with open(LIVE_TRACE_PATH) as f:
        trace_json = json.load(f)
    print(f"Using this session's live trace ({LIVE_TRACE_PATH})")
except Exception as e:
    print(f"Could not extract a live trace ({e}); using the bundled sample instead")
    with open(SAMPLE_TRACE_PATH) as f:
        trace_json = json.load(f)

### Reading a Langfuse Trace as JSON

Here are a few tips and tricks to explore your trace data if you have extracted it as a JSON file.

#### Understanding the Trace Structure

A Langfuse trace JSON contains several key sections:

##### 1. **Top-Level Trace Information**
- `id`: Unique identifier for the trace
- `name`: Name of the traced operation (e.g., "LangGraph")
- `timestamp`: When the trace was created
- `latency`: Total execution time in seconds

##### 2. **Input and Output (Messages)**

The trace's `input` and `output` fields contain the conversation flow:

**Input** (`trace.input`):
```json
{
  "messages": [
    "What is the weather in Miami?"
  ]
}
```
This represents the initial user query.

**Output** (`trace.output`):
```json
{
  "messages": [
    {...},  // Human message
    {...},  // AI message with tool call
    {...},  // Tool response
    {...}   // Final AI response
  ]
}
```

The output contains a **list of messages** representing the complete conversation flow:

1. **Human Message** (`type: "human"`):
   - Contains the user's question
   - Has a unique `id` for reference

2. **AI Message with Tool Call** (`type: "ai"`):
   - The model's decision to use a tool
   - `tool_calls`: Array of tools the model wants to invoke
   - `response_metadata`: Contains token usage and model information

3. **Tool Message** (`type: "tool"`):
   - Contains the tool's execution result
   - `content`: The actual data returned by the tool
   - `tool_call_id`: Links back to the tool call that triggered it

     To find the output of a tool_call execution, locate the observation who's output message has a tool_call_id equal to the id value of the tool_call you are interested in.


4. **Final AI Message** (`type: "ai"`):
   - The model's final response using the tool results
   - `content`: Human-readable answer
   - `finish_reason: "stop"`: Indicates completion

##### 3. **Observations**

The `observations` array (`trace.observations`) contains detailed execution steps:

- **Types**: `GENERATION`, `SPAN`, `CHAIN`, `EVENT`
- **Key Fields**:
  - `name`: Operation name (e.g., "route_tools", "llm")
  - `input`: Data sent to this step
  - `output`: Data returned from this step
  - `metadata`: Additional context (LangGraph step info, node names, etc.)
  - `startTime` / `endTime`: Timing information
  - `parentObservationId`: Links to parent observation for hierarchy

#### Reading the Message Flow

To understand the execution flow:

1. Start with `trace.input.messages` for the initial query
2. Follow `trace.output.messages` sequentially to see:
   - User question → AI tool call → Tool execution → AI final answer
3. Cross-reference with `trace.observations` for detailed execution metadata
4. Match tool calls using `tool_call_id` to link requests and responses

In [ ]:
def extract_tool_calls(trace_data):
    """
    Extract tool calls and their corresponding results from a Langfuse trace.

    Tool calls are found in AI messages that have a non-empty 'tool_calls' list.
    Tool call results are matched by tool_call_id, searching first in the trace
    output messages, then falling back to the observations.

    Returns a list of dicts with keys: name, args, tool_call_id, result
    """
    messages = trace_data.get("output", {}).get("messages", [])
    observations = trace_data.get("observations", [])

    # Collect all tool calls from AI messages
    tool_calls = []
    for msg in messages:
        if msg.get("type") in ("ai", "assistant") and msg.get("tool_calls"):
            for tc in msg["tool_calls"]:
                tool_calls.append({
                    "name": tc["name"],
                    "args": tc["args"],
                    "tool_call_id": tc["id"],
                    "result": None,
                })

    for tc in tool_calls:
        tid = tc["tool_call_id"]

        for msg in messages:
            if msg.get("type") == "tool" and msg.get("tool_call_id") == tid:
                tc["result"] = msg.get("content")
                break

        if tc["result"] is None:
            for obs in observations:
                obs_output = obs.get("output", {}) or {}
                obs_meta = obs.get("metadata", {}) or {}
                if (obs_output.get("tool_call_id") == tid
                        or obs_meta.get("tool_call_id") == tid):
                    tc["result"] = obs_output.get("content")
                    break

    return tool_calls

In [ ]:
tool_calls = extract_tool_calls(trace_json)

for tc in tool_calls:
    print(f"Tool:    {tc['name']}")
    print(f"Args:    {tc['args']}")
    print(f"Call ID: {tc['tool_call_id']}")
    print(f"Result:  {tc['result']}")
    print()

## 5. Conclusion

In this notebook, we added observability to a granite agent using **Langfuse tracing** by:

- **Reusing the function-calling agent** built in 01, via `granite_agent.agent.build_agent()`, equipped with stock price and weather tools.
- **Instrumenting the agent** with Langfuse's callback handler, which automatically captured traces from LangChain's execution hierarchy: chains, LLM generations, and tool calls, without any code changes to the agent itself.
- **Collecting and exploring a trace** programmatically via the Langfuse API, examining the JSON structure to understand the full lifecycle of an agent request -- and doing so from a bundled sample trace even when Langfuse isn't reachable.

Tracing is a foundational step toward reliable agent development. With traces, you can track unexpected agent trajectories, measure latency across steps, and identify which tool calls or LLM generations need improvement.

To go further, explore the **Advanced Topics** section below, which covers:

- **Experiments** — systematically evaluate agent outputs against expected results with automated scoring.
- **Manual tracing** — gain fine-grained control over trace structure when automatic instrumentation isn't sufficient.

## 6. ⭐ Advanced topics (stretch)

### Create a Langfuse experiment to evaluate the agent output

Experiment tracking should be a foundational practice to build into your agent workflow from day one.
[(IBM, Agent Observability and Operations)](https://www.ibm.com/downloads/documents/us-en/1443d5dd174f42e6)

Langfuse experiments are used to loop your agent through a dataset and optionally apply Evaluation Methods to the results :

1. Create dataset items with inputs and expected outputs
2. Define your task function to test
3. Write evaluators to score results
4. Run experiment to get automated scores across all test cases

| Component | Description | Example |
|-----------|-------------|---------|
| **Dataset** | Collection of test cases | defined as `local_data` in this notebook with stock price questions |
| **Dataset Item** | Single test case with input and optional expected output | `{"input": "What were...", "expected_output": "On September..."}` |
| **Task** | Application code being tested | `agent_execution_to_evaluate()` - executes your agent |
| **Evaluator** | Function that scores outputs | `accuracy_evaluator()` - checks if expected output appears in response |
| **Score** | Evaluation result (numeric/categorical/boolean) | `1.0` (correct) or `0.0` (incorrect) |
| **Experiment Run** | Execution of task on all dataset items | `langfuse_client.run_experiment()` |

**Note:** without a real `AV_STOCK_API_KEY`, `get_stock_price` always returns the same fixed demo value regardless of the date asked, so the accuracy score below will be low no matter how well the agent reasons -- that's expected, not a bug in your setup.

In [ ]:
def agent_execution_to_evaluate(*, item, **kwargs):
    user_message = HumanMessage(item["input"])
    response = agent.invoke({"messages": [user_message]})
    return response.get('messages')[-1].content

local_data = [
    {
    "input": "What were the IBM stock prices on September 2, 2026?",
    "expected_output": "on september 2 2026, ibm’s stock traded between **$256.64** (low) and **$264.66** (high)."
    }
]

# Define evaluation functions
def accuracy_evaluator(*, input, output, expected_output, metadata, **kwargs):
    if not expected_output:
        return Evaluation(name="accuracy", value=0.0, comment="No expected output provided")

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = scorer.score(expected_output, output)

    rouge_l_fmeasure = scores['rougeL'].fmeasure

    return Evaluation(
        name="accuracy",
        value=rouge_l_fmeasure,
        comment=f"ROUGE-1: {scores['rouge1'].fmeasure:.3f} | ROUGE-L: {scores['rougeL'].fmeasure:.3f}"
    )


def length_evaluator(*, input, output, **kwargs):
    return Evaluation(name="response_length", value=len(output), comment=f"Response has {len(output)} characters")

result = langfuse_client.run_experiment(
    name="Stock Market Quizz",
    description="Testing basic functionality",
    data=local_data,
    task=agent_execution_to_evaluate,
    evaluators=[accuracy_evaluator, length_evaluator]
)

After running the experiment, you can view the results in the Langfuse dashboard.

In the above example our experiment tracked two key metrics:

1. **Accuracy**: A ROUGE-L score evaluating how closely the granite response matches the expected answer, based on the longest common subsequence between the two texts.

2. **Response Length**: The character count of each response, helping assess verbosity.

#### Review the experiment in Langfuse UI

*(Open the experiment run link printed above to see the per-item scores in the Langfuse dashboard.)*

### Send a Trace to Langfuse Manually

#### Managing Langfuse observations with a simple nested trace

In this example, we manually create a nested trace using context managers. The root observation is a chain that contains a generation and a tool call as children.

In [ ]:
with langfuse_client.start_as_current_observation(
    as_type="chain",
    name="example_agent",
    input={"query": "What's the weather in San Francisco?"}
) as root:

    with langfuse_client.start_as_current_observation(
        as_type="generation",
        name="llm",
        input={"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]},
        model=model_path
    ) as gen:
        gen.update(
            output={"content": "I'll check the weather in San Francisco."},
            usage={"input": 10, "output": 5, "total": 15}
        )
    with langfuse_client.start_as_current_observation(
        as_type="tool",
        name="weather_tool",
        input={"location": "San Francisco"}
    ) as tool:
        tool.update(output={"temp": 72, "conditions": "sunny"})

    root.update(output={"response": "It's 72°F and sunny"})

langfuse_client.flush()

print(f"Trace ID: {root.trace_id}")

#### Managing Langfuse observations for LangGraph events

In this section we stream our function calling LangGraph agent execution and log each step : Granite calls and tool executions are tracked as nested observations within a single Langfuse trace named ```Simple_Agent_Trace```.

In [ ]:
class LangfuseObservationTracker:
    """
    Uses langfuse_client.start_as_current_observation() to create child
    observations. Nesting is automatic through OpenTelemetry context
    propagation — any observation created within the root trace's context
    manager becomes a child of that trace.
    """

    def __init__(self, langfuse_client):
        """
        Initialize the tracker with a Langfuse client.

        Args:
            langfuse_client: The Langfuse client instance used to create observations
        """
        self.langfuse_client = langfuse_client

    def track_llm_call(self, name: str, input_messages: Any, output: Any, model: Optional[str] = None):
        """
        Track an LLM call as a Langfuse generation.

        A generation is a specialized observation type for AI model interactions.
        It captures the input prompt, output response, and model information.

        Args:
            name: Name of the LLM call (e.g., "ChatModel")
            input_messages: The input messages sent to the LLM
            output: The response from the LLM
            model: Optional model name (e.g., "ibm/granite-4-h-small")
        """
        print(f"  Tracking LLM call: {name}")

        with self.langfuse_client.start_as_current_observation(
            as_type="generation",
            name=name,
            input=input_messages,
            model=model,
        ) as generation:
            generation.update(output=output)
        return generation

    def track_tool_call(self, name: str, tool_input: Any, tool_output: Any):
        """
        Track a tool execution as a Langfuse tool observation.

        A tool observation captures when the agent calls an external tool/function.
        It records what tool was called, with what input, and what it returned.

        Args:
            name: Name of the tool (e.g., "get_current_weather")
            tool_input: The input arguments passed to the tool
            tool_output: The result returned by the tool
        """
        print(f"  Tracking tool call: {name}")

        with self.langfuse_client.start_as_current_observation(
            as_type="tool",
            name=name,
            input=tool_input,
        ) as tool_obs:
            tool_obs.update(output=tool_output)
        return tool_obs

In [ ]:
def test_event_generator(graph: CompiledStateGraph, user_input: str):
    """
    manual Langfuse tracing from our langgraph graph execution.

    Steps:
    1. Create root trace - container for all observations, represents entire agent execution
    2. Stream agent events - capture step-by-step graph node executions
    3. Log observations - LLM calls as "generation", tool calls as "tool"
    4. Flush to Langfuse - ensure all data is sent to server
    """

    user_message = HumanMessage(user_input)
    input_state = State(messages=[user_message])
    session_id = str(uuid.uuid4())

    print(f"User input: {user_input}")
    print(f"Session ID: {session_id}\n")

    print("STEP 1: Creating root trace...")

    with langfuse_client.start_as_current_observation(
        as_type="chain",
        name="Simple_Agent_Trace",
        input={"user_message": user_input},
        metadata={"session_id": session_id}
    ) as trace:
        print(f"Trace created with ID: {trace.trace_id}\n")

        tracker = LangfuseObservationTracker(langfuse_client)

        print("STEP 2: Executing agent and capturing events...\n")

        final_output = None
        llm_call_count = 0
        tool_call_count = 0

        # event is a dict with node_name as key and state update as value
        for event in graph.stream(input_state):
            for node_name, state_update in event.items():
                print(f"Node executed: {node_name}")

                if node_name == "model":
                    llm_call_count += 1
                    messages = state_update.get("messages", [])
                    if messages:
                        last_message = messages[-1]

                        input_msgs = input_state.get("messages", []) if llm_call_count == 1 else []
                        output_content = last_message.content if hasattr(last_message, "content") else str(last_message)

                        tracker.track_llm_call(
                            name=f"LLM_Call_{llm_call_count}",
                            input_messages=input_msgs,
                            output={"content": output_content},
                            model=model_path
                        )

                        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
                            print(f"  LLM decided to call {len(last_message.tool_calls)} tool(s)")
                        else:
                            print(f"  LLM generated final response")

                elif node_name == "tools":
                    tool_call_count += 1
                    messages = state_update.get("messages", [])
                    if messages:
                        for msg in messages:
                            if hasattr(msg, "name"):
                                tool_name = msg.name
                                tool_output = msg.content if hasattr(msg, "content") else str(msg)
                                tracker.track_tool_call(
                                    name=tool_name,
                                    tool_input={"call": f"Tool call #{tool_call_count}"},
                                    tool_output={"result": tool_output}
                                )
                                print(f"  Tool '{tool_name}' executed")
                final_output = state_update

        if final_output and "messages" in final_output:
            final_message = final_output["messages"][-1]
            final_response = final_message.content if hasattr(final_message, "content") else str(final_message)
        else:
            final_response = "No response generated"

        print("\nSTEP 4: Finalizing trace...")
        trace.update(output={"final_response": final_response})

    langfuse_client.flush()
    print("Trace data sent to Langfuse\n")

    print("-" * 80)
    print("EXECUTION SUMMARY:")
    print(f"  LLM calls: {llm_call_count}")
    print(f"  Tool calls: {tool_call_count}")
    print(f"  Final response: {final_response[:100]}...")
    print(f"  Trace ID: {trace.trace_id}")
    print(f"  View in Langfuse: {os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')}/trace/{trace.trace_id}")
    print("=" * 80 + "\n")

    return {
        "session_id": session_id,
        "output": final_response,
        "trace_id": trace.trace_id,
        "llm_calls": llm_call_count,
        "tool_calls": tool_call_count
    }

In [ ]:
user_input = "What is the weather in Miami?"
test_event_generator(graph=agent, user_input=user_input)

The resulting ```Simple_Agent_Trace``` trace in Langfuse shows the full agent execution as a nested hierarchy:

```Simple_Agent_Trace → LLM_Call_1 → get_current_weather → LLM_Call_2```

*(Screenshot omitted here -- open the trace URL printed above to see the nested hierarchy for yourself.)*

## Reuse in later notebooks

Everything reused in this notebook comes from `src/granite_agent/`, built in 01:

| Reusable | From |
| --- | --- |
| `granite_agent.model.get_llm()` | 01, Step 2 |
| `granite_agent.tools.get_stock_price`, `granite_agent.tools.get_current_weather` | 01, Step 4 |
| `granite_agent.agent.State` | 01, Step 6 |
| `granite_agent.agent.build_agent()`, `granite_agent.agent.run_agent()` | 01, Steps 7-8 |

No new reusable functions were added here -- this notebook is about *observing* the existing agent, not extending it.